In [0]:
df = spark.table("hdb_resale_prices.bronze.raw_datasets").toPandas()
df.head()

In [0]:
df.shape # (986276, 14)

# check for null
df.isnull().sum().sort_values(ascending = False)
    # Only remaining_lease have 709050 rows with null (Only recent datasets has remaining_lease)

# check fr duplicates
df.duplicated().sum()
    # 1929 true duplicates

# Check str formatting
df['town'].value_counts(dropna = False) # no issue with town naming
df['flat_type'].value_counts(dropna = False) 
# Change in wordings here -> Need to remediate
    # MULTI GENERATION       279
    # MULTI-GENERATION       275
df["storey_range"].value_counts(dropna=False)
# There are overlaps in storey_range but inconclusive

In [0]:
# standardize all to MULTI-GENERATION
df['flat_type'] = df['flat_type'].str.replace("MULTI GENERATION", "MULTI-GENERATION")

df['flat_type'].value_counts(dropna = False) # check

# Remove exact duplicates
business_key = ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease']

df_dedup = df.drop_duplicates(subset=business_key, keep="first")
df_dedup.duplicated().sum() # no duplicates


In [0]:
spark.createDataFrame(df_dedup).write.mode("overwrite").saveAsTable("hdb_resale_prices.silver.resale_prices")